# 08 · E3-A: placement and integration → decision D-E3

**Runtime:** T4 GPU. **Time:** about 30–45 minutes. **Resumable.** Protocol v2 encoding.

**Question.** Where in the LM, and in what form, are optical features most useful?

**Design** (protocol E3 Part A, Core). Layer ∈ {L3, L6, L12} (25 / 50 / 100% depth) × pooling ∈ {mean, last token}
× integration ∈ {replacement r, parallel [h; r]} = 12 cells. Maps: F2-fourier (best optical variant in E2 v2) and
H (best digital), plus identity A at each layer/pooling. D = 4d, N = 100 (10 paired subsamples) and full data,
3 map seeds, same tuning budget as E2.

**Decision rule (fixed before running).** D-E3 = the cell with the best N = 100 dev accuracy for F2-fourier;
ties go to replacement, then mean pooling, then the later layer. Written to `reports/e3_decision.json` and
frozen for E4 onward.

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "surensagh/PhotonicsAI-Project"                    # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!python scripts/run_e2.py --config configs/e3.yaml

In [ ]:
print(open(os.path.join(WORKSPACE, "reports", "e3_table.md"), encoding="utf-8").read())

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(WORKSPACE, "reports", "e3_fig.png")))